# 03 · Input pipeline check (plan step 2.5)

Checks the `tf.data` pipeline (`plantdx.data.pipeline`) on the real images with the settings of
`configs/base.yaml`:

1. one batch from each split: shapes, dtype, value range, one-hot labels;
2. augmentation examples on real leaves → `reports/figures/aug_examples.jpg` (look at them!);
3. speed: first training epoch (fills the disk cache), later epochs from the cache, augmentation on the GPU,
   and the val set before and after caching;
4. memory and disk used by the caches.

Results are written to `reports/pipeline_check.json`. Select a **T4 GPU** runtime (the speed numbers
are meant for the training setup). About 10 minutes.

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # the Colab copy is disposable: always match GitHub exactly (a plain pull can be blocked
    # by files that an earlier run of a notebook wrote into the repo folder)
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
head = !git -C {REPO_DIR} rev-parse HEAD
remote = !git -C {REPO_DIR} rev-parse origin/main
assert head == remote, "repo is not at the latest commit: run this cell again"
!git -C {REPO_DIR} log -1 --oneline
!nvidia-smi -L

## 2 · Images

In [ ]:
from plantdx.data.images import download_archive, extract_variant

VARIANT, DATA_ROOT = "color", "/content/data"
print("new files extracted:", extract_variant(download_archive(), DATA_ROOT, VARIANT))

## 3 · Datasets from the config and one batch per split

In [ ]:
import json
import numpy as np
import tensorflow as tf

from plantdx.config import load_config, set_seed
from plantdx.data.pipeline import datasets_from_config, load_split

cfg = load_config("configs/base.yaml")
set_seed(cfg["seed"])
ds = datasets_from_config(cfg, DATA_ROOT)
batch_size = cfg["train"]["batch_size"]

check = {"tensorflow": tf.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")],
         "batch_size": batch_size, "image_size": cfg["data"]["image_size"], "splits": {}}
for split in ("train", "val", "test"):
    n = len(load_split(split))
    x, y = next(iter(ds[split]))
    info = {
        "images": n,
        "batches_per_epoch": int(np.ceil(n / batch_size)),
        "x_shape": list(x.shape), "x_dtype": x.dtype.name,
        "x_min": float(tf.reduce_min(x)), "x_max": float(tf.reduce_max(x)),
        "y_shape": list(y.shape), "y_rows_sum_to_1": bool(np.allclose(y.numpy().sum(1), 1)),
        "classes_in_first_batch": int(len(set(y.numpy().argmax(1)))),
    }
    check["splits"][split] = info
    print(split, info)

assert check["splits"]["train"]["x_shape"] == [batch_size, 224, 224, 3]
assert all(s["x_min"] >= 0 and s["x_max"] <= 255 and s["y_rows_sum_to_1"] for s in check["splits"].values())
print("\nPASS: shapes, value range and labels")

## 4 · Augmentation examples ← **look at this figure**

Each row: the image as the model sees it at evaluation, then 5 training versions. Check that the leaf and its
lesions stay recognisable (no lesion cut away, no unnatural colours) and that rotated corners are filled
plausibly.

In [ ]:
from pathlib import Path
from IPython.display import Image as IPImage, display

from plantdx.data.pipeline import AugmentConfig
from plantdx.viz import plot_augmentations

FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
out = plot_augmentations(load_split("train"), DATA_ROOT, VARIANT, FIG / "aug_examples.jpg",
                         augment=AugmentConfig.from_dict(cfg["augment"]), n_images=6, n_aug=5, seed=cfg["seed"])
display(IPImage(str(out)))

## 5 · Speed and memory

The training split is cached on the VM's local disk (`data.cache_train: disk`) and augmentation runs on the
GPU as the first model layers (`train.augment_on: model`), because doing it in tf.data on Colab's 2 CPU cores
gave only 67 images/s (first run of this notebook, D-008). Measured here:

* **epoch 1**: decode + write the disk cache (once per VM);
* **epoch 2+**: read from the cache;
* **GPU augmentation**: the augmentation layers applied to cached batches on the GPU;
* val before/after its in-memory cache, and the RAM/disk the caches use.

In [ ]:
import os
import shutil
import time

import psutil

from plantdx.data.pipeline import augmentation_from_config
from plantdx.paths import cache_dir

def timed_pass(dataset, max_batches=None):
    t0, n = time.perf_counter(), 0
    for i, (x, _) in enumerate(dataset):
        n += int(x.shape[0])
        if max_batches and i + 1 >= max_batches:
            break
    return n, time.perf_counter() - t0

speed = {}
n, s = timed_pass(ds["train"])                          # full pass: fills the disk cache
speed["train_epoch1_fill_cache"] = {"images": n, "seconds": round(s, 1), "images_per_s": round(n / s, 1)}
n, s = timed_pass(ds["train"], max_batches=150)         # from the cache
speed["train_from_cache"] = {"images": n, "seconds": round(s, 1), "images_per_s": round(n / s, 1)}

aug = augmentation_from_config(cfg)
apply_aug = tf.function(lambda x: aug(x, training=True))
batches = [x for x, _ in ds["train"].take(20)]
with tf.device("/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"):
    apply_aug(batches[0])                                # trace + warm up
    t0 = time.perf_counter()
    for _ in range(3):
        for x in batches:
            out = apply_aug(x)
    _ = out.numpy()
    s = time.perf_counter() - t0
speed["gpu_augmentation"] = {"images": 3 * sum(int(b.shape[0]) for b in batches), "seconds": round(s, 2)}
speed["gpu_augmentation"]["images_per_s"] = round(speed["gpu_augmentation"]["images"] / s, 1)

rss0 = psutil.Process().memory_info().rss
n, first = timed_pass(ds["val"])
_, second = timed_pass(ds["val"])
rss1 = psutil.Process().memory_info().rss
speed["val_pass_seconds"] = {"first_fills_cache": round(first, 1), "second_from_cache": round(second, 1)}
speed["val_cache_ram_gb"] = round((rss1 - rss0) / 1e9, 2)

n_train = check["splits"]["train"]["images"]
bottleneck = min(speed["train_from_cache"]["images_per_s"], speed["gpu_augmentation"]["images_per_s"])
speed["input_epoch_seconds_after_cache"] = round(n_train / bottleneck, 1)
speed["disk_cache_gb"] = round(sum(f.stat().st_size for f in cache_dir().rglob("*") if f.is_file()) / 1e9, 2)
speed["disk_free_gb"] = round(shutil.disk_usage(cache_dir()).free / 1e9, 1)
speed["ram_total_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
speed["cpu_count"] = os.cpu_count()
speed["before_fix_cpu_augmentation_images_per_s"] = 67.5  # first run of this notebook, 18 Nov 2026
check["speed"] = speed

print(json.dumps(speed, indent=2))
_ = Path("reports/pipeline_check.json").write_text(json.dumps(check, indent=2) + "\n")

**How to read it.** `input_epoch_seconds_after_cache` is how long one training epoch takes if the model
itself were free; the slower of "from cache" and "GPU augmentation" decides it. Epoch 1 is slower once per VM
because it decodes every JPEG. If the VM restarts, the disk cache is gone and epoch 1 is slow again.

## 6 · Download the results

In [ ]:
from google.colab import files

!rm -f /content/step2_pipeline.zip && zip -q /content/step2_pipeline.zip reports/pipeline_check.json reports/figures/aug_examples.jpg
files.download("/content/step2_pipeline.zip")